# Task 1 — TinyStories checkpoint demo

ECE 661 Team 32 · Phase 1 · Will Ding

Loads an official TinyStories checkpoint, generates a story, and saves a structured record.

**This notebook contains no model-loading or generation logic of its own** — it calls
`tinystories_project`, the same code `scripts/run_task1.py` and every later task use.
Fix a bug in the package, not here.

Prerequisite: `pip install -e ".[dev,notebook]"` and select `.venv` as the kernel.

## 1. Imports and environment

In [ ]:
from tinystories_project.config import find_project_root, load_task1_config
from tinystories_project.runtime import available_devices, describe_device, resolve_device
from tinystories_project.inference import generate_stories, load_causal_lm
from tinystories_project.inference.interface import GenerationParams
from tinystories_project.records import build_records, read_jsonl, write_jsonl

print('available devices:', available_devices())
print('auto resolves to :', describe_device(resolve_device('auto')))

## 2. Load the experiment config

Settings come from `configs/task1.yaml`, never from numbers typed into this notebook —
so a result here is reproducible by `scripts/run_task1.py` with the same file.

In [ ]:
cfg = load_task1_config(find_project_root() / 'configs' / 'task1.yaml')

print('run name :', cfg.run_name)
print('model    :', cfg.model.model_id)
print('seed     :', cfg.seed)
print('params   :', cfg.generation.to_dict())
print('prompts  :')
for p in cfg.prompts:
    print('  -', p)

## 3. Load the checkpoint

`load_causal_lm` resolves the device (CUDA → MPS → CPU), sets the missing pad token,
forces left padding for correct batched generation, puts the model in eval mode, and
measures parameter counts.

First run downloads ~20 MB. The `UNEXPECTED` keys in the load report are obsolete
causal-mask buffers from transformers 4.28 — harmless, not learned weights.

In [ ]:
bundle = load_causal_lm(
    cfg.model.model_id,
    tokenizer_id=cfg.model.tokenizer_id,
    device=cfg.model.device,
    dtype=cfg.model.dtype,
)
bundle

### Measured model metadata

Note the gap between the two parameter counts: the checkpoint is named "1M" but measures
~3.75M total, because the 50,257-token embedding table dominates a model this small.
We always report **measured** counts. This distinction decides what
"parameter-matched" means for the Phase 2 comparison.

In [ ]:
m = bundle.metadata
for k, v in m.to_dict().items():
    if isinstance(v, int):
        v = f'{v:,}'
    print(f'{k:<28} {v}')

## 4. Generate

`generate_stories` works on any object satisfying the `TextGenerator` protocol —
a `ModelBundle` here, but equally Rachel's minGRU or Luel's Amharic model.

In [ ]:
stories = generate_stories(
    bundle,
    cfg.prompts,
    params=cfg.generation,
    seed=cfg.seed,
    batch_size=cfg.batch_size,
)

print(f'{len(stories)} completions generated')

## 5. Prompt vs. generated continuation

The prompt and the new text are kept strictly separate — `completion` never echoes the
prompt. Tasks 3 (ROUGE) and 4 (the judge) depend on this.

In [ ]:
for s in stories:
    print('=' * 78)
    print(f'PROMPT       [{s.prompt_index}.{s.sample_index}]: {s.prompt}')
    print(f'CONTINUATION ({s.n_new_tokens} new tokens, finish={s.finish_reason}):')
    print(s.completion.strip())
print('=' * 78)

### Reproducibility check

Same seed ⇒ same text, given the same model, params, prompt order, batch size, and device.

In [ ]:
a = generate_stories(bundle, cfg.prompts[:1], max_new_tokens=40, seed=1234)
b = generate_stories(bundle, cfg.prompts[:1], max_new_tokens=40, seed=1234)
c = generate_stories(bundle, cfg.prompts[:1], max_new_tokens=40, seed=4321)

print('seed 1234 == seed 1234 :', a[0].completion == b[0].completion)
print('seed 1234 == seed 4321 :', a[0].completion == c[0].completion)

## 6. Save structured output

One JSONL line per completion (schema v1.0). Later tasks read this exact format and add
their scores under `extra` rather than reshaping the record.

In [ ]:
run_id = f'{cfg.run_name}-notebook'
records = build_records(stories, bundle.metadata, cfg.generation, run_id=run_id)

out_path = find_project_root() / 'results' / 'task1' / 'task1_demo_notebook.jsonl'
write_jsonl(records, out_path)
print('wrote', len(records), 'records ->', out_path)

In [ ]:
# Read back and confirm the round-trip
restored = list(read_jsonl(out_path))
r = restored[0]

print('records read :', len(restored))
print('record_id    :', r.record_id)
print('prompt_id    :', r.prompt_id)
print('model        :', r.model['model_id'])
print('params total :', f"{r.model['n_parameters_total']:,}")
print('seed         :', r.seed)
print('schema       :', r.schema_version)
print('full_text == prompt + completion:', r.full_text == r.prompt + r.completion)

## 7. Next steps

Task 1 is complete. Still to do in Phase 1:

- **Task 2** — reproduce Figures 9–11 and add our 20 prompts
- **Task 3** — ROUGE on train vs. validation (Section 4.4.1, Figures 14–17)
- **Task 4** — the Gemini judge (10 completions/prompt at T=1, graded at T=0).
  **Not yet implemented, and the course API must not be called until it is started** —
  it spends a shared $250 team budget.

See `CLAUDE.md` for the record schema, reproducibility notes, and standards.